# Local Robotics RAG Assistant — Learning & Evaluation Notebook

This notebook explains the complete project in a learning-oriented way.

**Final flow**

`Documents → Extraction → Chunking → Embeddings → FAISS + BM25 → CrossEncoder → Filtering → Ollama → Answer + Sources`

The notebook is for learning, experimentation, evaluation, and interview preparation.  
The production application remains in `src/` and `app.py`.

## 1. What is RAG?

**RAG = Retrieval-Augmented Generation.**

Instead of asking an LLM to answer only from its internal training, we first retrieve relevant information from our own knowledge base.

```text
User Question
     ↓
Retrieve relevant chunks
     ↓
Pass those chunks to the LLM
     ↓
Generate a grounded answer
     ↓
Show answer + sources
```

Our knowledge base contains AMR documentation, ROS 2 Humble docs, Jetson Orin Nano docs, RPLIDAR C1 docs, ESP32-C3 docs, IMX219 camera docs, BTS7960 docs, and audio notes.

## 2. Project setup

This notebook should be saved inside:

```text
Local_RAG_Robotics_Assistant/
└── notebooks/
    └── Local_Robotics_RAG_Learning_Notebook.ipynb
```

The following cell makes the project root importable.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 3. Import the production modules

In [ ]:
from src.document_loader import load_documents
from src.text_splitter import create_chunks
from src.embeddings import EmbeddingModel
from src.vector_store import load_vector_store, search_vector_store
from src.retriever import HybridRetriever
from src.rag_pipeline import RoboticsRAG

print("Imports successful.")

## 4. Load the robotics knowledge base

The loader supports PDF, DOCX, TXT, RST, and HTML.

Every source is normalized into a dictionary like:

```python
{
    "text": "...",
    "source": "...",
    "source_path": "...",
    "category": "...",
    "file_type": "...",
    "page": ...
}
```

This metadata is what later lets us show document names and PDF page numbers.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw"

documents = load_documents(DATA_PATH)

print("Document sections:", len(documents))
print("Total characters :", f"{sum(len(d['text']) for d in documents):,}")

### Development result

The current project produced roughly:

- **22 source files**
- **223 document sections**
- **555,748 extracted characters**

PDFs are stored page-by-page where possible.

In [ ]:
from collections import Counter

print("Categories:")
for k, v in Counter(d["category"] for d in documents).items():
    print(f"  {k:<20} {v}")

print("\nFile types:")
for k, v in Counter(d["file_type"] for d in documents).items():
    print(f"  {k:<20} {v}")

## 5. Inspect extracted text

In [ ]:
sample = documents[0]

print("Source   :", sample["source"])
print("Category :", sample["category"])
print("Type     :", sample["file_type"])
print("Page     :", sample["page"])
print("\nPreview:\n")
print(sample["text"][:1200])

## 6. Chunking

Large documents are divided into smaller overlapping pieces.

Current configuration:

```python
chunk_size = 1000
chunk_overlap = 200
```

Overlap helps preserve information that crosses a chunk boundary.

In [ ]:
chunks = create_chunks(
    documents,
    chunk_size=1000,
    chunk_overlap=200,
)

avg_size = sum(len(c["text"]) for c in chunks) / len(chunks)

print("Document sections :", len(documents))
print("Generated chunks  :", len(chunks))
print("Average size      :", round(avg_size, 1))

After the word-boundary fix, the project produced about **939 clean chunks**.  
The exact count may change if documents or chunking rules change.

In [ ]:
for chunk in chunks[:3]:
    print("=" * 70)
    print("Chunk ID :", chunk["chunk_id"])
    print("Source   :", chunk["source"])
    print("Page     :", chunk["page"])
    print()
    print(chunk["text"][:700])
    print()

## 7. Embeddings

The project uses:

`sentence-transformers/all-MiniLM-L6-v2`

Each chunk becomes a **384-dimensional vector** representing semantic meaning.

In [ ]:
embedding_model = EmbeddingModel()

sample_texts = [
    "The robot uses LiDAR to detect obstacles.",
    "Laser scanning helps the robot detect nearby objects.",
    "The motor driver controls the wheel motors.",
]

sample_embeddings = embedding_model.encode_documents(sample_texts)

print("Embedding matrix shape:", sample_embeddings.shape)

## 8. FAISS semantic search

FAISS searches dense vectors by semantic similarity.

This works especially well for conceptual questions where the question and answer do not use exactly the same words.

In [ ]:
faiss_index, stored_chunks = load_vector_store(
    PROJECT_ROOT / "vector_db"
)

print("Vectors in FAISS :", faiss_index.ntotal)
print("Chunk metadata   :", len(stored_chunks))

In [ ]:
question = "What does the AMR watchdog monitor?"

query_embedding = embedding_model.encode_query(question)

results = search_vector_store(
    index=faiss_index,
    chunks=stored_chunks,
    query_embedding=query_embedding,
    top_k=3,
)

for rank, result in enumerate(results, 1):
    print("=" * 70)
    print("Rank  :", rank)
    print("Score :", round(result["score"], 4))
    print("Source:", result["source"])
    print()
    print(result["text"][:700])

## 9. Why FAISS alone was not enough

The query:

> What is the maximum range of the RPLIDAR C1?

initially found the correct datasheet but not the exact specification chunk.

Technical tables contain short phrases, model names, and numbers, so semantic similarity alone can miss the most useful passage.

That motivated adding **BM25 keyword retrieval**.

## 10. BM25 + FAISS = Hybrid Retrieval

FAISS is good at **meaning**.

BM25 is good at **exact words**, such as:

- RPLIDAR
- C1
- BEST_EFFORT
- model names
- technical terms
- numbers

The production retriever combines both approaches.

In [ ]:
hybrid_retriever = HybridRetriever(
    index=faiss_index,
    chunks=stored_chunks,
    embedding_model=embedding_model,
)

print("Hybrid retriever ready.")

## 11. CrossEncoder reranking

FAISS and BM25 generate candidates quickly.

A CrossEncoder then evaluates each question + candidate pair together:

```text
Question + Candidate
        ↓
CrossEncoder
        ↓
Relevance score
```

The project uses:

`cross-encoder/ms-marco-MiniLM-L6-v2`

In [ ]:
question = "What is the maximum measuring range of the RPLIDAR C1?"

results = hybrid_retriever.search(question, top_k=5)

for rank, result in enumerate(results, 1):
    print("=" * 70)
    print("Rank           :", rank)
    print("Rerank score   :", round(result["rerank_score"], 4))
    print("Semantic score :", round(result["semantic_score"], 4))
    print("BM25 score     :", round(result["keyword_score"], 4))
    print("Source         :", result["source"])
    print("Page           :", result["page"])
    print()
    print(result["text"][:800])

### Important result

After reranking, the RPLIDAR query correctly returned information such as:

> RPLIDAR C1 has a measuring distance of up to a radius of 12 meters.

So the information was present. The original problem was **ranking**, not extraction.

## 12. Question-only filtering

The AMR handbooks also contain interview-question lists.

A chunk containing:

> Why is BEST_EFFORT QoS appropriate for camera images?

can rank extremely high because it nearly matches the user query word-for-word, even though it contains no answer.

Therefore the production RAG pipeline filters chunks that are mostly question lists.

**Key lesson:** query relevance is not always the same as answer usefulness.

In [ ]:
def looks_like_question_only_chunk(text):
    return text.count("?") >= 3 and "answer:" not in text.lower()

count = sum(
    looks_like_question_only_chunk(c["text"])
    for c in stored_chunks
)

print("Question-list-like chunks:", count)

## 13. Local LLM with Ollama

Models tested:

- Qwen3 0.6B
- Gemma3 1B
- Qwen3 1.7B

Final model:

**Qwen3 1.7B**

Current generation configuration:

```text
temperature = 0
context     = 2048
keep_alive  = 2 minutes
top context = 3 chunks
```

The laptop has an NVIDIA MX330 with 2 GB VRAM, so Ollama performs split CPU/GPU inference.

## 14. Full RAG pipeline

This initializes the complete system:

```text
Question
→ FAISS + BM25
→ CrossEncoder
→ Question-only filter
→ Top 3 contexts
→ Grounded prompt
→ Ollama / Qwen3 1.7B
→ Answer + sources
```

> Ollama must be running locally before executing this cell.

In [ ]:
rag = RoboticsRAG(
    model_name="qwen3:1.7b"
)

print("Robotics RAG initialized.")

In [ ]:
question = "What does the AMR watchdog node monitor?"

result = rag.ask(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
seen = set()

for source in result["sources"]:
    key = (source["source"], source["page"])
    if key in seen:
        continue
    seen.add(key)

    if source["page"] is None:
        print("-", source["source"])
    else:
        print(f"- {source['source']} — Page {source['page']}")

## 15. Hallucination / refusal test

A grounded assistant should not invent undocumented hardware.

Example:

> What GPS module does my AMR use?

Expected response:

`I do not have enough information in the knowledge base to answer that.`

In [ ]:
question = "What GPS module does my AMR use?"

result = rag.ask(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(result["answer"])

## 16. Evaluation methodology

The evaluation set contains:

- **12 answerable questions**
- **3 intentionally unanswerable questions**

It covers:

- AMR-specific behavior
- ROS 2
- hardware specifications
- unsupported project details

Automatic evaluation checks source retrieval, expected answer terms, and refusal behavior.

However, automatic keyword scoring is imperfect, so final evaluation should combine:

`Automatic evaluation + Manual technical review`

In [ ]:
with open(
    PROJECT_ROOT / "evaluation_questions.json",
    "r",
    encoding="utf-8",
) as f:
    evaluation_questions = json.load(f)

print("Questions:", len(evaluation_questions))

for q in evaluation_questions[:5]:
    print(f"{q['id']:>2}. [{q['type']}] {q['question']}")

## 17. Model comparison

| Metric | Qwen3 1.7B | Gemma3 1B |
|---|---:|---:|
| Automatic score | **14/15 (93.3%)** | 10/15 (66.7%) |
| Retrieval source hits | **12/12** | **12/12** |
| Exact refusals | **3/3** | 0/3 |
| Fluency | Better | Weaker |
| Resource use | Higher | Lower |
| Final choice | **Qwen3 1.7B** | Comparison |

### Main conclusion

Both models received the correct source for all 12 answerable questions.

Therefore, many failures were caused by **generation quality**, not retrieval quality.

In [ ]:
# Optional: inspect saved evaluation CSVs.

import pandas as pd

results_dir = PROJECT_ROOT / "evaluation_results"

for filename in [
    "qwen3_1.7b_results.csv",
    "gemma3_1b_results.csv",
]:
    path = results_dir / filename

    print("\n", filename)

    if path.exists():
        df = pd.read_csv(path)
        display(df.head())
    else:
        print("Not found.")

## 18. Final architecture

```text
                     ROBOTICS DOCUMENTS
                            │
                            ↓
                  Multi-format Loader
              PDF / DOCX / RST / HTML / TXT
                            │
                            ↓
                       Text Cleaning
                            │
                            ↓
                  Overlapping Chunking
                 ~1000 chars / 200 overlap
                            │
                            ↓
                 SentenceTransformer
                  all-MiniLM-L6-v2
                            │
                            ↓
                    384-D Embeddings
                            │
                            ↓
              ┌─────────────┴─────────────┐
              ↓                           ↓
            FAISS                        BM25
      Semantic Retrieval          Keyword Retrieval
              │                           │
              └─────────────┬─────────────┘
                            ↓
                     Candidate Chunks
                            │
                            ↓
                       CrossEncoder
                        Reranking
                            │
                            ↓
                  Question-only Filter
                            │
                            ↓
                     Top 3 Contexts
                            │
                            ↓
                  Grounded RAG Prompt
                            │
                            ↓
                  Ollama / Qwen3 1.7B
                            │
                            ↓
                   Answer + Sources
                            │
                            ↓
                    Streamlit Dashboard
```

## 19. Key learnings

1. **RAG is more than FAISS.** Good RAG requires ingestion, retrieval, ranking, generation, grounding, and evaluation.
2. **FAISS and BM25 solve different problems.** Semantic search helps with concepts; BM25 helps with exact technical terms.
3. **Retrieval quality and generation quality are separate.** Both final models achieved 12/12 source hits, but their answers differed.
4. **Reranking matters.** It improved datasheet retrieval where the correct source was found but the wrong passage ranked highest.
5. **Query relevance is not answer usefulness.** Interview-question chunks matched perfectly but contained no answer.
6. **Local LLM size matters.** Smaller models were lighter but showed weaker grounded synthesis or refusal behavior.
7. **Unknown questions must be tested.** GPS, IMU, and microphone questions test hallucination control.
8. **Automatic evaluation is not enough.** Keyword scoring should be supplemented with technical manual review.

## 20. Interview-ready explanation

> I built a local robotics knowledge assistant using Retrieval-Augmented Generation. I ingest robotics documentation in multiple formats, normalize it, and split it into overlapping chunks. I generate dense embeddings using Sentence Transformers and index them with FAISS. For retrieval I combine FAISS semantic search with BM25 lexical search, then rerank the candidates using a CrossEncoder. I also filter question-only chunks before passing the best context to a local Ollama LLM. I evaluated the system on 15 questions covering project-specific knowledge, ROS 2, hardware specifications, and unsupported queries. Qwen3 1.7B achieved a 93.3% automatic score, with correct-source retrieval on all 12 answerable questions and correct refusal on all 3 unsupported questions.

## 21. Notebook vs production code

### Notebook
Use it for:

- understanding the pipeline
- experiments
- demonstrations
- evaluation interpretation
- interview revision

### Production files

```text
src/document_loader.py
src/text_splitter.py
src/embeddings.py
src/vector_store.py
src/retriever.py
src/reranker.py
src/llm.py
src/rag_pipeline.py

ingest.py
search.py
chat.py
evaluate.py
app.py
```

The Streamlit app is the user-facing application.